# PEFT Ablation Study — Colab training runner

Runs ONE training method (full fine-tune, one LoRA rank, or QLoRA) end-to-end on a Colab GPU:
trains under the project's fixed compute budget, evaluates the resulting checkpoint (task EM/F1 +
MMLU forgetting benchmark), and pushes the small `results/*.json` files back to GitHub.

This notebook contains **no training logic of its own** — it just clones the repo and runs the
same `scripts/train.py` / `scripts/evaluate.py` already validated locally on CPU. Re-run this whole
notebook once per method across Steps 4–6 (6 times total), changing only the `CONFIG` variable below.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU` (must be a GPU runtime, not CPU).
2. Add a GitHub token via the **Secrets** panel (key/padlock icon in the left sidebar):
   name it `GITHUB_TOKEN`, enable notebook access. Use a **fine-grained PAT scoped only to the
   `peft-ablation-study` repo** with Contents: Read/Write — not a broad classic token — since it's
   only needed to clone this one private repo and push the small results files back.
3. Set `CONFIG` in the cell below to the method you're running this time.

In [ ]:
# <-- CHANGE THIS for each of the 6 runs across Steps 4-6:
CONFIG = "configs/full_finetune.yaml"

# Options across the whole project:
#   configs/full_finetune.yaml   (Step 4)
#   configs/lora_rank4.yaml      (Step 5)
#   configs/lora_rank8.yaml      (Step 5)
#   configs/lora_rank16.yaml     (Step 5)
#   configs/lora_rank32.yaml     (Step 5)
#   configs/qlora.yaml           (Step 6)

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > select 'T4 GPU', then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Liang714/peft-ablation-study.git"

!git clone {REPO_URL} repo
%cd repo
!git config user.email "erhfwuliang@live.com"
!git config user.name "Liang714"
!git log --oneline -5

In [ ]:
# Colab already ships a CUDA-linked torch satisfying requirements.txt's torch>=2.1, so this
# should mostly skip reinstalling it. If torch.cuda.is_available() becomes False after this cell,
# restart the runtime (Runtime > Restart session) and re-run WITHOUT this install cell.
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Train

Runs under the project's fixed compute budget (`configs/base.yaml`'s `compute_budget.total_tokens`,
currently 2,383,278 tokens = 3 epochs over the real train split) — identical token budget for every
method, which is the entire point of this comparison. Logs training wall-clock time and peak GPU
memory, saves the checkpoint to `checkpoints/<run_name>/` (gitignored — stays in this Colab session's
storage, download it manually below if you want to keep it beyond this session).

In [ ]:
!PYTHONPATH=. python scripts/train.py --config {CONFIG}

## Evaluate the resulting checkpoint

Task EM/F1 on the held-out test split + MMLU forgetting-benchmark accuracy (the fixed, committed
150-question subset — same questions as the Step 3 baseline, so the forgetting delta computed in
Step 7 is meaningful).

In [ ]:
import json
from src.config import load_config

cfg = load_config(CONFIG)
RUN_NAME = cfg.run_name
CHECKPOINT_PATH = f"checkpoints/{RUN_NAME}"
print("run_name:", RUN_NAME, "| checkpoint:", CHECKPOINT_PATH)

In [ ]:
!PYTHONPATH=. python scripts/evaluate.py --config {CONFIG} --checkpoint {CHECKPOINT_PATH}

In [ ]:
# Quick sanity-check print of this run's headline numbers before pushing.
with open(f"results/{RUN_NAME}_train.json") as f:
    train_result = json.load(f)
with open(f"results/{RUN_NAME}_eval.json") as f:
    eval_result = json.load(f)

print("=== Training cost ===")
print("wall_clock_seconds:", train_result["wall_clock_seconds"])
print("peak_memory_mb:", train_result["peak_memory_mb"])
print("train_loss:", train_result["train_loss"], "| val_loss:", train_result["val_loss"])
print()
print("=== Task + forgetting metrics ===")
print("task EM/F1:", eval_result["task_metrics"])
print("MMLU accuracy:", eval_result["mmlu"]["accuracy"])

## Push results back to GitHub

Only the small `results/*.json` files — checkpoints stay gitignored, as already set up in the repo.

In [ ]:
!git add results/*.json
!git commit -m "Colab run: {RUN_NAME} training + eval results"
!git push

## (Optional) Download the checkpoint

Only needed if you want to keep the trained weights beyond this Colab session (e.g. for the Step 9
inference endpoint). LoRA/QLoRA adapters are small (~5-15MB); the full fine-tune checkpoint is ~1-2GB
and slow to download — consider copying it to Google Drive instead if you need to keep it.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(f"/content/{RUN_NAME}_checkpoint", "zip", CHECKPOINT_PATH)
files.download(zip_path)